# T9 It1: offline red team of the recommended decision (T7 judge trained on NUDT only)

**Question:** is the decision independent of the input photos, i.e. would it stay safe (0 wrong answers) and useful on drone photos unlike
the ones it was built on? This cannot be *proven* from data; it can only be attacked. Five attacks with the data in hand (official 379 and NUDT
707 are both outside the judge's training data; the judge is trained on NUDT 101-606, cut 0.9 PC / 0.925 Jetson, as pre-registered in T8):

1. **How different do the inputs look?** A classifier that tells NUDT candidates from official ones (adversarial validation): if it can, the
   judge is extrapolating on official; which features differ most.
2. **Is P(right) still honest off its training data?** Wrong rate among candidates by P band, NUDT (out-of-fold) vs official vs 707.
3. **Where does it break by height?** Score / wrong per height band on official, judge vs today's rule.
4. **How close to a wrong answer?** Wrong answers if the cut were lower: the margin to the first failure.
5. **Which signal shifts would create wrong answers?** Shift one feature of every candidate by +-1 standard deviation (as a different camera,
   map or matcher might) and count wrong answers. P can only rise with inliers, z, hull and coverage, so lowering those is safe by
   construction; the other features are where a shift could push a wrong place over the cut.

In [ ]:
import os, sys, time, json
os.environ.setdefault("OMP_NUM_THREADS", "8")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
from train.checks.candidate_judge import FEATURES_CTX, INCREASING, baseline, decide, judge_probability, summary, labelled
from train.checks.holdout import NUDT_DESIGN, load_t8_table
RES = ROOT / "tests/Test_Iteration_9/results"
table = load_t8_table(ROOT, locked=False, blind=True)
table.loc[table.data.str.startswith("official"), "data"] = "official"      # both halves: none of official is in the judge's training data
EVAL = ["official", "nudt707"]
t0 = time.time(); P, MODEL = judge_probability(table, NUDT_DESIGN, FEATURES_CTX, "gbm"); table["p"] = P
print("judge trained on", NUDT_DESIGN, "%.0f s" % (time.time() - t0))
PC = dict(override=12, min_choice=3, wide_tau=0.95); TAU_PC, TAU_JET = 0.9, 0.925

## 1. How different do NUDT and official candidates look to the judge?

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
lab = table[labelled(table) & table.data.isin(NUDT_DESIGN + ["official"])]
X, y = lab[FEATURES_CTX].to_numpy(float), (lab.data == "official").to_numpy()
pr = cross_val_predict(HistGradientBoostingClassifier(max_depth=3, max_iter=200), X, y, cv=5, method="predict_proba")[:, 1]
print("all features: AUC NUDT vs official = %.3f (0.5 = cannot tell them apart)" % roc_auc_score(y, pr))
rows = []
for f in FEATURES_CTX:
    a = roc_auc_score(y, lab[f].to_numpy(float)); rows.append(dict(feature=f, auc=round(max(a, 1 - a), 3),
        nudt_median=lab.loc[~y, f].median(), official_median=lab.loc[y, f].median(), monotone="up" if f in INCREASING else ""))
SHIFT = pd.DataFrame(rows).sort_values("auc", ascending=False); SHIFT.to_csv(RES / "it1_feature_shift.csv", index=False); SHIFT.round(3)

## 2. Is P(right) honest off the training data? (labelled candidates with >= 3 inliers)

In [ ]:
bins = [0, 0.5, 0.8, 0.9, 0.95, 0.98, 1.0001]
L = table[labelled(table)].assign(band=lambda d: pd.cut(d.p, bins, right=False), wrong=lambda d: d.err_sigma > 8,
                                  group=lambda d: np.where(d.data.isin(NUDT_DESIGN), "NUDT design (out-of-fold)", d.data))
CAL = L.groupby(["group", "band"], observed=True).agg(n=("wrong", "size"), wrong=("wrong", "sum"), wrong_rate=("wrong", "mean")).reset_index()
CAL.to_csv(RES / "it1_calibration.csv", index=False)
CAL.pivot(index="band", columns="group", values=["n", "wrong", "wrong_rate"]).round(4)

## 3. By height band on official (all 379 photos; PC and Jetson)

In [ ]:
off = table[table.data.isin(EVAL)]
R = {("pc", "base"): baseline(off, 8, 11, min_choice=3), ("pc", "judge"): decide(off, off.p.to_numpy(), TAU_PC, **PC),
     ("jetson", "base"): baseline(off, 8, 8, jetson=True, m=5), ("jetson", "judge + early"): decide(off, off.p.to_numpy(), TAU_JET, jetson=True, m=5, early_tau=0.98, **PC)}
h = off.groupby(["data", "photo"]).height.first()
rows = []
for (var, name), r in R.items():
    r = r.assign(h=[h[(d, p)] for d, p in zip(r.data, r.photo)])
    r["band"] = np.where(r.data == "nudt707", "NUDT 707", pd.cut(r.h, [0, 125, 175, 275, 400], labels=["100-125", "125-175", "175-275", "275-351"]).astype(str))
    for b, g in r.groupby("band"):
        rows.append(dict(var=var, rule=name, band=b, n=len(g), score=round(6 * g.pts.mean(), 2), wrong=int((g.pts < 0).sum()), answered=round((g.stage != "skip").mean(), 3)))
BAND = pd.DataFrame(rows); BAND.to_csv(RES / "it1_height_bands.csv", index=False)
BAND.pivot_table(index=["band"], columns=["var", "rule"], values=["score", "wrong"]).round(2)

## 4. Margin to the first wrong answer: the cut lowered step by step

In [ ]:
rows = []
for tau in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.925, 0.95]:
    for var, kw in [("pc", {}), ("jetson", dict(jetson=True, m=5))]:
        s = summary(decide(off, off.p.to_numpy(), tau, **PC, **kw))
        for d in EVAL: rows.append(dict(var=var, tau=tau, data=d, score=s.loc[d, "score"], wrong=s.loc[d, "wrong"]))
MARGIN = pd.DataFrame(rows); MARGIN.to_csv(RES / "it1_cut_margin.csv", index=False)
MARGIN.pivot_table(index="tau", columns=["var", "data"], values=["wrong", "score"]).round(2)

## 5. One feature shifted by +-1 standard deviation (of the NUDT design candidates)

In [ ]:
sd = table[table.data.isin(NUDT_DESIGN) & (table.inliers >= 3)][FEATURES_CTX].std()
usable = (off.inliers >= 3).to_numpy()
Xo = off[FEATURES_CTX].to_numpy(float)
rows = []
for j, f in enumerate(FEATURES_CTX):
    for sign in (-1, 1):
        X2 = Xo.copy(); X2[:, j] += sign * sd[f]
        p2 = np.zeros(len(off)); p2[usable] = MODEL.predict_proba(X2[usable])[:, 1]
        for var, tau, kw in [("pc", TAU_PC, {}), ("jetson", TAU_JET, dict(jetson=True, m=5))]:
            s = summary(decide(off, p2, tau, **PC, **kw))
            rows.append(dict(feature=f, shift=sign, var=var, monotone=f in INCREASING, **{f"wrong_{d}": int(s.loc[d, "wrong"]) for d in EVAL},
                             **{f"score_{d}": s.loc[d, "score"] for d in EVAL}))
STRESS = pd.DataFrame(rows); STRESS.to_csv(RES / "it1_feature_stress.csv", index=False)
bad = STRESS[(STRESS.wrong_official > 0) | (STRESS.wrong_nudt707 > 0)]
print("shifts that create wrong answers:", len(bad), "of", len(STRESS)); bad.round(2)

In [ ]:
STRESS.pivot_table(index="feature", columns=["var", "shift"], values=["wrong_official", "wrong_nudt707"]).astype(int)